<a href="https://colab.research.google.com/github/Shreya00320/AI_lab/blob/main/Unification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
UNIFICATION (First Order Logic)

Representation:
  - Variable : lowercase string          -> 'x', 'y', 'z'
  - Constant : Capitalised string        -> 'John', 'Anand', 'A'
  - Function / Predicate : tuple         -> ('Father', 'y')      = Father(y)
                                            ('Knows', 'John', 'x') = Knows(John, x)
"""


def is_variable(t):
    return isinstance(t, str) and t[0].islower()


def is_compound(t):
    return isinstance(t, tuple)


def to_str(t):
    if is_compound(t):
        return f"{t[0]}({', '.join(to_str(a) for a in t[1:])})"
    return str(t)


def subst(theta, t):
    """Apply substitution theta to term t."""
    if is_variable(t):
        return subst(theta, theta[t]) if t in theta else t
    if is_compound(t):
        return (t[0],) + tuple(subst(theta, a) for a in t[1:])
    return t


def show_theta(theta):
    if theta is None:
        return "FAIL"
    return "{" + ", ".join(f"{to_str(subst(theta, v))}/{k}"
                           for k, v in theta.items()) + "}"


def occurs_check(var, x, theta):
    """Does variable var occur inside x? (y with Father(y) -> True)"""
    if var == x:
        return True
    if is_variable(x) and x in theta:
        return occurs_check(var, theta[x], theta)
    if is_compound(x):
        return any(occurs_check(var, a, theta) for a in x[1:])
    return False


def unify_var(var, x, theta):
    if var in theta:
        return unify(theta[var], x, theta)
    if is_variable(x) and x in theta:
        return unify(var, theta[x], theta)
    if occurs_check(var, x, theta):
        return None                       # FAILURE
    new_theta = dict(theta)
    new_theta[var] = x                    # {x/var}
    return new_theta


def unify(x, y, theta=None):
    """Return MGU as a dict, or None if unification fails."""
    if theta is None:
        theta = {}
    if x == y:                            # identical -> NIL
        return theta
    if is_variable(x):
        return unify_var(x, y, theta)
    if is_variable(y):
        return unify_var(y, x, theta)
    if is_compound(x) and is_compound(y):
        if x[0] != y[0] or len(x) != len(y):   # predicate / argument count differ
            return None
        for a, b in zip(x[1:], y[1:]):         # match arguments one by one
            theta = unify(a, b, theta)
            if theta is None:
                return None
        return theta
    return None                           # two different constants


# ---------------- DEMO ----------------
if __name__ == "__main__":
    tests = [
        (("King", "x"), ("King", "Anand")),
        (("Knows", "John", "x"), ("Knows", "John", "Jane")),
        (("Knows", "John", "x"), ("Knows", "y", "OJ")),
        (("Knows", "John", "x"), ("Knows", "y", ("Mother", "y"))),
        (("Knows", "John", "x"), ("Knows", "x", "OJ")),                # fail
        (("Knows", ("Father", "y"), "y"), ("Knows", "x", "x")),        # occurs check
        (("Eats", "x", "Apple"), ("Eats", "Riya", "y")),
        (("p", "B", "x", ("f", ("g", "z"))), ("p", "z", ("f", "y"), ("f", "y"))),
    ]
    for a, b in tests:
        print(f"UNIFY({to_str(a)}, {to_str(b)}) = {show_theta(unify(a, b))}")

UNIFY(King(x), King(Anand)) = {Anand/x}
UNIFY(Knows(John, x), Knows(John, Jane)) = {Jane/x}
UNIFY(Knows(John, x), Knows(y, OJ)) = {John/y, OJ/x}
UNIFY(Knows(John, x), Knows(y, Mother(y))) = {John/y, Mother(John)/x}
UNIFY(Knows(John, x), Knows(x, OJ)) = FAIL
UNIFY(Knows(Father(y), y), Knows(x, x)) = FAIL
UNIFY(Eats(x, Apple), Eats(Riya, y)) = {Riya/x, Apple/y}
UNIFY(p(B, x, f(g(z))), p(z, f(y), f(y))) = {B/z, f(g(B))/x, g(B)/y}
